# Atelier : livrer le modèle affiné du service client

Le fil rouge de la formation : le service client de Biclou & Cie, dont le modèle doit trier chaque message (catégorie, priorité) et y répondre selon la charte, en un objet JSON. Le TP d'entraînement a produit un adaptateur QLoRA de Qwen3-1.7B, l'atelier d'évaluation l'a mesuré. Reste à le livrer : en faire un modèle que l'équipe peut servir sans `peft` ni GPU, et dont on saura, dans six mois, d'où il vient.

1. Fusionner l'adaptateur
2. Exporter en safetensors
3. Convertir en GGUF et quantifier
4. Servir avec llama-server
5. La fiche de traçabilité

L'adaptateur est le vôtre si vous l'avez publié à la fin du TP d'entraînement (le secret Colab `HF_TOKEN`), sinon un adaptateur de référence, entraîné par le même code.

Ce notebook demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Comptez une heure et demie, le code compris ; le calcul en prend une demi-heure environ (une estimation), surtout aux exercices 3 et 4, que llama.cpp fait sur le processeur, même sur GPU. Sans GPU, il tourne quand même, en plus petit : Qwen3-0.6B et son adaptateur au lieu de Qwen3-1.7B, sur 4 tickets de test au lieu de 150, en 25 minutes environ sur deux processeurs, comme ceux de Colab, dont une douzaine pour l'exercice 1 et six pour le 2 : sur processeur, `transformers` génère lentement, ce que l'exercice 4 compare à llama.cpp. Sur 4 tickets, les chiffres ne disent presque rien : c'est la méthode qui compte.

## Données et outils

`test.jsonl` : les 150 tickets de test validés par l'équipe, avec la réponse attendue. `llama.cpp/` : le dépôt de llama.cpp à l'étiquette `b12359`, sa version 1.0.0 (13 novembre 2026), pour son script de conversion et sa bibliothèque Python `gguf` ; `llama-b12359/` : ses programmes compilés pour Linux, qui calculent sur le processeur (`llama-quantize`, `llama-server`).

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/test.jsonl
!git clone -q --depth 1 --branch b12359 https://github.com/ggml-org/llama.cpp
!wget -q https://github.com/ggml-org/llama.cpp/releases/download/b12359/llama-b12359-bin-ubuntu-x64.tar.gz
!tar -xzf llama-b12359-bin-ubuntu-x64.tar.gz
%pip install -q peft ./llama.cpp/gguf-py
%pip uninstall -q -y torchao  # la version de Colab est trop ancienne pour peft

In [ ]:
import collections
import gc
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request
from importlib.metadata import version

import gguf
import pandas as pd
import requests
import torch
from huggingface_hub import HfApi, ModelCard, ModelCardData
from openai import OpenAI
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU = torch.cuda.is_available()
# bfloat16 à partir des GPU Ampere (A100, L4, H100) ; le T4 calcule en float16.
BF16 = GPU and torch.cuda.get_device_capability()[0] >= 8
# Sur CPU, float32 : les demi-précisions y sont très lentes.
DTYPE = (torch.bfloat16 if BF16 else torch.float16) if GPU else torch.float32
DEVICE = "cuda" if GPU else "cpu"
MODELE = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
# L'étiquette de llama.cpp clonée et téléchargée ci-dessus, et ses programmes.
LLAMA_CPP = "b12359"
BINAIRES = f"llama-{LLAMA_CPP}"
print(MODELE, "sur GPU" if GPU else "sans GPU, sur 4 tickets")

tokenizer = AutoTokenizer.from_pretrained(MODELE)
tokenizer.padding_side = "left"
test = pd.read_json("test.jsonl", lines=True)
test_part = test[: None if GPU else 4]
print(len(test), "tickets de test ;", len(test_part), "mesurés ici")


def libere() -> None:
  """Rend au GPU la mémoire des objets supprimés par `del`."""
  gc.collect()
  if GPU:
    torch.cuda.empty_cache()


def taille_mo(chemin: str) -> float:
  """La taille d'un fichier, ou des fichiers d'un dossier, en Mo."""
  if os.path.isdir(chemin):
    return sum(taille_mo(os.path.join(chemin, f)) for f in os.listdir(chemin))
  return os.path.getsize(chemin) / 1e6


def execute(commande: list[str]) -> str:
  """Lance une commande et renvoie sa sortie ; si elle échoue, lève une erreur."""
  resultat = subprocess.run(commande, capture_output=True, text=True)
  sortie = resultat.stdout + resultat.stderr
  if resultat.returncode != 0:
    raise RuntimeError(f"{commande[0]} a échoué :\n{sortie[-3000:]}")
  return sortie

L'adaptateur, dans le dossier `ADAPTATEUR` : le vôtre, s'il est sur votre Hub et a été entraîné sur `MODELE`, sinon celui de référence, de [shuuchuu/models](https://github.com/shuuchuu/models).

In [ ]:
ADAPTATEUR = "support-sft-lora"
REFERENCE = "https://github.com/shuuchuu/models/raw/refs/heads/main/" + (
  "support-sft-lora-qwen3-1.7b" if GPU else "support-sft-lora-qwen3-0.6b"
)


def adaptateur_du_hub() -> str | None:
  """Télécharge votre adaptateur dans ADAPTATEUR, s'il existe pour MODELE."""
  jeton = os.environ.get("HF_TOKEN")
  try:
    from google.colab import userdata

    jeton = userdata.get("HF_TOKEN")
  except Exception:
    pass
  if not jeton:
    return None
  api = HfApi(token=jeton)
  depot = f"{api.whoami()['name']}/support-sft-lora"
  if not api.repo_exists(depot):
    return None
  with open(api.hf_hub_download(depot, "adapter_config.json")) as f:
    if json.load(f)["base_model_name_or_path"] != MODELE:
      return None
  api.snapshot_download(depot, local_dir=ADAPTATEUR)
  return depot


depot = adaptateur_du_hub()
if depot is None:
  os.makedirs(ADAPTATEUR, exist_ok=True)
  for fichier in ["adapter_config.json", "adapter_model.safetensors"]:
    urllib.request.urlretrieve(f"{REFERENCE}/{fichier}", f"{ADAPTATEUR}/{fichier}")
print("Adaptateur :", depot or REFERENCE)

## La tâche et sa mesure

Le code du TP de mesure, repris tel quel : `CONSIGNES`, la consigne courte que le modèle affiné a reçue à l'entraînement ; `evalue(sorties)`, qui mesure les sorties des premiers tickets de `test_part` (`json_valide`, `categorie`, `priorite`, `charte` pour les règles vérifiables de la charte, `mots`) ; `genere_tout(modele, conversations)`, qui génère par lots, sans échantillonnage ; et `conversation_seule(message)`, la consigne en message `system` et le message du client en message `user`.

In [ ]:
CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
CONSIGNES = f"""Tu es l'assistant du service client de Biclou & Cie, magasins de vélos.
Pour le message du client, réponds uniquement par un objet JSON avec trois clés :
- categorie : une valeur parmi {", ".join(CATEGORIES)} ;
- priorite : basse, normale ou haute ;
- reponse : la réponse à envoyer au client."""
SIGNATURE = "Bonne journée,\nL'équipe Biclou & Cie"
TUTOIEMENT = re.compile(r"\b(tu|te|toi|ton|ta|tes)\b|\bt['’]", re.IGNORECASE)


def lit_json(texte: str) -> dict:
  """L'objet JSON d'une réponse du modèle, ou {} s'il n'est pas valide."""
  texte = texte.strip().removeprefix("```json").removesuffix("```").strip()
  try:
    objet = json.loads(texte)
  except json.JSONDecodeError:
    return {}
  return objet if isinstance(objet, dict) else {}


def respecte_charte(reponse: str) -> bool:
  reponse = reponse.strip()
  return (
    reponse.startswith("Bonjour")
    and reponse.endswith(SIGNATURE)
    and not TUTOIEMENT.search(reponse)
    and 40 <= len(reponse.split()) <= 120
  )


def evalue(sorties: list[str]) -> pd.Series:
  objets = [lit_json(s) for s in sorties]
  reponses = [str(o.get("reponse", "")) for o in objets]
  return (
    pd.Series(
      {
        "json_valide": sum(
          {"categorie", "priorite", "reponse"} <= o.keys() for o in objets
        ),
        "categorie": sum(
          o.get("categorie") == c for o, c in zip(objets, test_part["categorie"])
        ),
        "priorite": sum(
          o.get("priorite") == p for o, p in zip(objets, test_part["priorite"])
        ),
        "charte": sum(respecte_charte(r) for r in reponses),
        "mots": sum(len(r.split()) for r in reponses),
      }
    )
    .div(len(sorties))
    .round(2)
  )


def genere_tout(
  modele, conversations: list[list[dict]], taille_lot: int = 16, max_tokens: int = 384
) -> list[str]:
  textes = []
  for debut in range(0, len(conversations), taille_lot):
    prompts = [
      tokenizer.apply_chat_template(
        c, tokenize=False, add_generation_prompt=True, enable_thinking=False
      )
      for c in conversations[debut : debut + taille_lot]
    ]
    entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      sorties = modele.generate(
        **entrees,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    textes += tokenizer.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
  return textes


def conversation_seule(message: str) -> list[dict]:
  return [
    {"role": "system", "content": CONSIGNES},
    {"role": "user", "content": message},
  ]

## 1. Fusionner l'adaptateur

Un adaptateur LoRA reste à côté des poids : chaque couche linéaire qu'il cible calcule $xW$ et, en plus, $\frac{\alpha}{r} xAB$. Le fusionner, c'est ajouter $\frac{\alpha}{r} AB$ à $W$ une fois pour toutes : il reste un modèle ordinaire, de l'architecture et de la taille du modèle de base, que tout outil sait charger, sans `peft`.

Chargez `MODELE` en `DTYPE` sur `DEVICE`, sans quantification, ajoutez-lui l'adaptateur du dossier `ADAPTATEUR`, en mode évaluation, et comptez ses paramètres. Générez avec `genere_tout` les sorties des conversations d'`essai` (ci-dessous), en mesurant le temps par ticket. Puis fusionnez l'adaptateur dans les poids, comptez de nouveau les paramètres, et générez de nouveau. Rassemblez dans un tableau les paramètres et le temps par ticket des deux modèles, et comptez les sorties identiques.

Pourquoi pas en NF4, comme à l'entraînement ? Fusionner dans des poids de 4 bits, c'est les déquantifier, leur ajouter la mise à jour, puis les quantifier de nouveau : l'arrondi sur 4 bits efface une partie de ce que l'adaptateur a appris, et `peft` le signale par un avertissement. On fusionne dans des poids en 16 bits (en float32 sans GPU), et l'on quantifie ensuite, si besoin, le modèle fusionné : c'est l'exercice 3.

Les deux modèles donnent-ils les mêmes sorties ? Que sont devenus les paramètres de l'adaptateur ? Quand garderiez-vous l'adaptateur séparé ?

**Documentation utile**

- [`PeftModel.from_pretrained`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.from_pretrained)
- [`merge_and_unload`](https://huggingface.co/docs/peft/package_reference/tuners#peft.tuners.tuners_utils.BaseTuner.merge_and_unload)

In [ ]:
# Les conversations de l'essai : 32 tickets sur GPU, les 4 de test_part sans GPU.
essai = [conversation_seule(m) for m in test_part["message"].head(32)]

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE)` : sans `quantization_config`, les poids en `DTYPE` ; `PeftModel.from_pretrained(base, ADAPTATEUR).eval()` y ajoute l'adaptateur, en mode évaluation
- Les paramètres d'un modèle : `sum(p.numel() for p in modele.parameters())`
- `modele.merge_and_unload()` ajoute les mises à jour de l'adaptateur aux poids et renvoie le modèle de base, un modèle `transformers` ordinaire : gardez-lui le nom `modele`, que les exercices suivants utilisent
- Le temps par ticket : `time.time()` avant et après `genere_tout`, divisé par `len(essai)`

```python
def compte_parametres(modele) -> int:
  return sum(p.numel() for p in modele.parameters())


base = AutoModelForCausalLM.from_pretrained(...)
modele = PeftModel.from_pretrained(...).eval()
parametres_lora = compte_parametres(modele)
debut = time.time()
sorties_lora = genere_tout(modele, essai)
s_lora = (time.time() - debut) / len(essai)

modele = ...
parametres = compte_parametres(modele)
debut = time.time()
sorties_fusion = genere_tout(modele, essai)
s_fusion = ...

identiques = sum(a == b for a, b in zip(sorties_lora, sorties_fusion))
print(f"Sorties identiques : {identiques} sur {len(essai)}")
pd.DataFrame(
  {"parametres": [parametres_lora, parametres], "s_par_ticket": [s_lora, s_fusion]},
  index=["adaptateur séparé", "fusionné"],
).round(2)
```

### Solution

In [ ]:
def compte_parametres(modele) -> int:
  return sum(p.numel() for p in modele.parameters())


base = AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE)
modele = PeftModel.from_pretrained(base, ADAPTATEUR).eval()
parametres_lora = compte_parametres(modele)
debut = time.time()
sorties_lora = genere_tout(modele, essai)
s_lora = (time.time() - debut) / len(essai)

modele = modele.merge_and_unload()
parametres = compte_parametres(modele)
debut = time.time()
sorties_fusion = genere_tout(modele, essai)
s_fusion = (time.time() - debut) / len(essai)

identiques = sum(a == b for a, b in zip(sorties_lora, sorties_fusion))
print(f"Sorties identiques : {identiques} sur {len(essai)}")
pd.DataFrame(
  {"parametres": [parametres_lora, parametres], "s_par_ticket": [s_lora, s_fusion]},
  index=["adaptateur séparé", "fusionné"],
).round(2)

# Les paramètres de l'adaptateur (1 à 2 % de plus) sont passés dans les poids : le
# modèle fusionné a exactement ceux du modèle de base, et plus aucun calcul à côté de
# chaque couche, un gain qui dépend du matériel et de la taille des lots. Les
# sorties sont les mêmes, à l'arrondi près : en 16 bits, W + AB arrondi diffère un
# peu de xW + xAB calculés à part, et une génération gloutonne peut bifurquer sur
# quelques tickets. On garde l'adaptateur séparé pour servir plusieurs adaptateurs
# sur un seul modèle de base (vLLM, LoRAX en chargent à la demande), ou pour en
# changer sans redéployer le modèle.

## 2. Exporter en safetensors

Enregistrez le modèle fusionné et son tokenizer dans le dossier `EXPORT` (ci-dessous). Affichez ses fichiers et leurs tailles, face à ceux du dossier de l'adaptateur. Le *chat template*, le gabarit qui met une conversation en forme de prompt (les balises de Qwen3, le bloc de réflexion vide de `enable_thinking=False`), est-il dans l'export ? Que se passerait-il, côté serveur, sans lui ?

Puis vérifiez que l'export se suffit à lui-même : libérez le modèle, rechargez le modèle et le tokenizer depuis `EXPORT` seul, avec `transformers` (sans `peft` ni `MODELE`), et mesurez-le sur `test_part` avec la consigne courte, dans `sorties_export`. Rangez sa mesure dans un tableau `comparatif`, une ligne `transformers`, avec le temps par ticket et la taille du dossier en Mo. Les sorties de l'essai sont-elles celles du modèle fusionné ? Les mesures, proches de celles du modèle affiné dans l'atelier d'évaluation ?

**Documentation utile**

- [`PreTrainedModel.save_pretrained`](https://huggingface.co/docs/transformers/main_classes/model#transformers.PreTrainedModel.save_pretrained)
- [Les chat templates](https://huggingface.co/docs/transformers/chat_templating)

In [ ]:
EXPORT = "support-sft-fusionne"

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `modele.save_pretrained(EXPORT)`, puis `tokenizer.save_pretrained(EXPORT)` ; `taille_mo(chemin)` (cellule du début) : la taille d'un fichier ou d'un dossier
- Le chat template s'enregistre dans `chat_template.jinja`
- `del modele, base`, puis `libere()` : un seul modèle en mémoire à la fois
- `AutoTokenizer.from_pretrained(EXPORT)`, puis `tokenizer.padding_side = "left"` : `genere_tout` lit la variable `tokenizer`, gardez-lui ce nom ; `AutoModelForCausalLM.from_pretrained(EXPORT, dtype=DTYPE, device_map=DEVICE)`
- `evalue(sorties)` renvoie une `pd.Series` : ajoutez-lui le temps et la taille, puis `pd.DataFrame({"transformers": mesures}).T`, une ligne

```python
modele.save_pretrained(EXPORT)
tokenizer.save_pretrained(...)
for dossier in [ADAPTATEUR, EXPORT]:
  fichiers = {f: round(taille_mo(os.path.join(dossier, f)), 1) for f in os.listdir(...)}
  print(dossier, fichiers)
print("Chat template exporté :", os.path.exists(...))

del modele, base
libere()
tokenizer = AutoTokenizer.from_pretrained(...)
tokenizer.padding_side = "left"
modele = AutoModelForCausalLM.from_pretrained(...)
debut = time.time()
sorties_export = genere_tout(
  modele, [conversation_seule(m) for m in test_part["message"]]
)
mesures = evalue(sorties_export)
mesures["s_par_ticket"] = round((time.time() - debut) / len(test_part), 2)
mesures["taille_mo"] = round(taille_mo(EXPORT))
memes = sum(a == b for a, b in zip(sorties_fusion, sorties_export))
print(f"Mêmes sorties que le modèle fusionné : {memes} sur {len(essai)}")
comparatif = pd.DataFrame({"transformers": ...}).T
comparatif
```

### Solution

In [ ]:
modele.save_pretrained(EXPORT)
tokenizer.save_pretrained(EXPORT)
for dossier in [ADAPTATEUR, EXPORT]:
  fichiers = {
    f: round(taille_mo(os.path.join(dossier, f)), 1) for f in os.listdir(dossier)
  }
  print(dossier, fichiers)
print(
  "Chat template exporté :", os.path.exists(os.path.join(EXPORT, "chat_template.jinja"))
)

del modele, base
libere()
tokenizer = AutoTokenizer.from_pretrained(EXPORT)
tokenizer.padding_side = "left"
modele = AutoModelForCausalLM.from_pretrained(EXPORT, dtype=DTYPE, device_map=DEVICE)
debut = time.time()
sorties_export = genere_tout(
  modele, [conversation_seule(m) for m in test_part["message"]]
)
mesures = evalue(sorties_export)
mesures["s_par_ticket"] = round((time.time() - debut) / len(test_part), 2)
mesures["taille_mo"] = round(taille_mo(EXPORT))
memes = sum(a == b for a, b in zip(sorties_fusion, sorties_export))
print(f"Mêmes sorties que le modèle fusionné : {memes} sur {len(essai)}")
comparatif = pd.DataFrame({"transformers": mesures}).T
comparatif

# L'export pèse le modèle de base entier, 2 octets par paramètre en 16 bits (4 en
# float32), quelques dizaines de fois l'adaptateur. Il garde le chat template
# (chat_template.jinja) : sans lui, un serveur mettrait les messages dans un gabarit
# par défaut, ou n'en mettrait aucun, et le modèle recevrait un prompt qu'il n'a
# jamais vu à l'entraînement. C'est une erreur de livraison fréquente, et silencieuse :
# le modèle répond, mais moins bien.

## 3. Convertir en GGUF et quantifier

GGUF, le format de llama.cpp (et d'Ollama ou de LM Studio, qui s'appuient sur lui), tient en un seul fichier : les poids, le tokenizer, le chat template et la description de l'architecture. Ses poids se quantifient après l'entraînement, par blocs : Q8_0 garde 8 bits par poids et un facteur d'échelle par bloc de 32 poids ; Q4_K_M, environ 4 bits par poids, par super-blocs de 256, et 6 bits pour quelques tenseurs plus sensibles (le « M », *medium*).

Avec les outils de llama.cpp, à l'étiquette `LLAMA_CPP` :

1. convertissez `EXPORT` en un fichier GGUF en F16 (16 bits), avec le script `llama.cpp/convert_hf_to_gguf.py` ;
2. quantifiez ce fichier en Q8_0, puis en Q4_K_M, avec le programme `llama-quantize`, dans le dossier `BINAIRES`.

Les chemins des trois fichiers sont dans `ggufs` (ci-dessous). Faites le tableau de leurs tailles en Mo, l'export safetensors compris, avec les bits par paramètre (8 fois la taille en octets, divisé par `parametres`, de l'exercice 1). Puis lisez les métadonnées du fichier Q4_K_M avec `gguf.GGUFReader` : l'architecture, le type de fichier, la présence du chat template, les réglages d'échantillonnage (`general.sampling.*`), et le nombre de tenseurs par type de quantification.

Q4_K_M fait-il vraiment 4 bits par paramètre ? Tous les tenseurs sont-ils quantifiés ? D'où viennent les réglages d'échantillonnage, et qui s'en servira ?

**Documentation utile**

- [La conversion et la quantification avec llama.cpp](https://github.com/ggml-org/llama.cpp/blob/b12359/tools/quantize/README.md)
- [Le format GGUF](https://huggingface.co/docs/hub/gguf)
- [`gguf.GGUFReader`](https://github.com/ggml-org/llama.cpp/blob/b12359/gguf-py/gguf/gguf_reader.py)

In [ ]:
# Tout se passe sur disque : le modèle chargé n'est plus utile.
del modele
libere()
os.makedirs("gguf", exist_ok=True)
ggufs = {
  "F16": "gguf/support-sft-f16.gguf",
  "Q8_0": "gguf/support-sft-q8_0.gguf",
  "Q4_K_M": "gguf/support-sft-q4_k_m.gguf",
}

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `execute(commande)` (cellule du début) lance une commande, une liste de chaînes, et lève une erreur avec la fin de sa sortie si elle échoue
- La conversion : `[sys.executable, "llama.cpp/convert_hf_to_gguf.py", EXPORT, "--outfile", ggufs["F16"], "--outtype", "f16"]` ; `sys.executable`, le Python du notebook, qui a `gguf`
- La quantification : `[f"{BINAIRES}/llama-quantize", ggufs["F16"], ggufs[nom], nom]`, pour `nom` dans `"Q8_0"`, `"Q4_K_M"`
- `lecteur = gguf.GGUFReader(chemin)` ; `lecteur.fields[cle].contents()`, la valeur de la métadonnée `cle` (`"general.architecture"`, `"general.file_type"`) ; `gguf.LlamaFileType(numero).name`, le nom d'un type de fichier ; `lecteur.tensors`, dont chacun a un `name` et un `tensor_type.name`, que `collections.Counter` compte
- Sans GPU, la quantification en Q4_K_M prend une minute environ, sur deux processeurs

```python
execute([sys.executable, "llama.cpp/convert_hf_to_gguf.py", ...])
for nom in ["Q8_0", "Q4_K_M"]:
  execute([f"{BINAIRES}/llama-quantize", ...])

tailles = {"safetensors": taille_mo(EXPORT)} | {n: taille_mo(c) for n, c in ...}
pd.DataFrame(
  {
    "taille_mo": tailles,
    "bits_par_parametre": {n: 8e6 * t / parametres for n, t in tailles.items()},
  }
).round(1)

lecteur = gguf.GGUFReader(ggufs["Q4_K_M"])
print("Architecture :", lecteur.fields["general.architecture"].contents())
type_fichier = lecteur.fields[...].contents()
print("Type de fichier :", gguf.LlamaFileType(type_fichier).name)
print("Chat template :", ... in lecteur.fields)
echantillonnage = {
  cle: champ.contents()
  for cle, champ in lecteur.fields.items()
  if cle.startswith("general.sampling")
}
print("Échantillonnage :", echantillonnage)
print("Tenseurs par type :", collections.Counter(...))
```

### Solution

In [ ]:
execute(
  [
    sys.executable,
    "llama.cpp/convert_hf_to_gguf.py",
    EXPORT,
    "--outfile",
    ggufs["F16"],
    "--outtype",
    "f16",
  ]
)
for nom in ["Q8_0", "Q4_K_M"]:
  execute([f"{BINAIRES}/llama-quantize", ggufs["F16"], ggufs[nom], nom])

tailles = {"safetensors": taille_mo(EXPORT)} | {
  n: taille_mo(c) for n, c in ggufs.items()
}
print(
  pd.DataFrame(
    {
      "taille_mo": tailles,
      "bits_par_parametre": {n: 8e6 * t / parametres for n, t in tailles.items()},
    }
  ).round(1)
)

lecteur = gguf.GGUFReader(ggufs["Q4_K_M"])
print("Architecture :", lecteur.fields["general.architecture"].contents())
type_fichier = lecteur.fields["general.file_type"].contents()
print("Type de fichier :", gguf.LlamaFileType(type_fichier).name)
print("Chat template :", "tokenizer.chat_template" in lecteur.fields)
echantillonnage = {
  cle: champ.contents()
  for cle, champ in lecteur.fields.items()
  if cle.startswith("general.sampling")
}
print("Échantillonnage :", echantillonnage)
print(
  "Tenseurs par type :",
  collections.Counter(t.tensor_type.name for t in lecteur.tensors),
)

# Q4_K_M dépasse 4 bits par paramètre : les facteurs d'échelle de chaque bloc, et les
# tenseurs gardés en 6 bits (Q6_K), dont l'embedding, lourd sur un petit modèle au
# vocabulaire de 152 000 tokens. Les normalisations restent en float32 : peu de
# poids, très sensibles. Le fichier porte le chat template, et les réglages
# d'échantillonnage de generation_config.json (température 0,6, celle que Qwen
# recommande) : un serveur les applique quand une requête ne donne pas les siens.

## 4. Servir avec llama-server

`llama-server` sert un fichier GGUF par une API HTTP compatible avec celle d'OpenAI : une application qui appelle déjà un modèle avec le client `openai` n'a qu'à changer d'adresse. Les programmes téléchargés calculent sur le processeur : c'est la cible d'une livraison sans GPU, un poste ou un petit serveur.

Écrivez `demarre_serveur(chemin)`, qui lance `llama-server` en arrière-plan sur le fichier GGUF `chemin`, sur le port `PORT`, sous le nom `"support-sft"`, avec autant de threads que de processeurs (`CPUS`) et son journal dans le fichier `llama-server.log`, puis attend que son point `/health` réponde avec le code 200, et renvoie le processus. Écrivez `genere_serveur(conversations)`, qui envoie chaque conversation au serveur avec `client`, le client `openai` (ci-dessous), sans échantillonnage, 384 tokens au plus, la réflexion de Qwen3 désactivée comme dans `genere_tout`, et renvoie les textes.

Puis, pour le Q8_0 et le Q4_K_M : démarrez le serveur ; vérifiez qu'il met une conversation en forme comme `transformers`, grâce à son point `/apply-template`, qui renvoie le prompt qu'il construirait ; générez les sorties de `serveur_part`, avec la consigne courte, en mesurant le temps par ticket ; puis arrêtez le serveur. Rassemblez dans un tableau `formats`, une ligne par format, la ligne `transformers` comprise, sur les tickets de `serveur_part` : la mesure d'`evalue`, la part de sorties identiques à celles de `transformers` (`sorties_export`), le temps par ticket et la taille du fichier.

Qu'a coûté la quantification, en qualité, en taille, en temps ? Lequel des deux fichiers livreriez-vous ?

**Documentation utile**

- [`llama-server`](https://github.com/ggml-org/llama.cpp/blob/b12359/tools/server/README.md)
- [`subprocess.Popen`](https://docs.python.org/3/library/subprocess.html#subprocess.Popen)
- [Le client `openai`, les complétions de chat](https://platform.openai.com/docs/api-reference/chat/create)

In [ ]:
PORT = 8642
URL = f"http://localhost:{PORT}"
# Les processeurs dont dispose le notebook : 2 sur Colab.
CPUS = len(os.sched_getaffinity(0))
# Les tickets que sert llama-server, sur le processeur : 16 sur GPU, les 4 sans.
serveur_part = test_part.head(16)
client = OpenAI(base_url=f"{URL}/v1", api_key="aucune")

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Le serveur : `subprocess.Popen([f"{BINAIRES}/llama-server", "--model", chemin, "--port", str(PORT), "--alias", "support-sft", "--threads", str(CPUS)], stdout=journal, stderr=subprocess.STDOUT)`, avec `journal = open("llama-server.log", "w")`
- `requests.get(f"{URL}/health")` lève `requests.ConnectionError` tant que le serveur n'écoute pas, puis répond 503 pendant qu'il charge le modèle : réessayez chaque seconde ; `serveur.poll()` vaut `None` tant que le processus vit, sinon le serveur s'est arrêté, et `llama-server.log` dit pourquoi
- `client.chat.completions.create(model="support-sft", messages=conversation, temperature=0, max_tokens=384, extra_body=...)` ; `extra_body` ajoute à la requête des champs que l'API d'OpenAI n'a pas : `{"chat_template_kwargs": {"enable_thinking": False}}`, que le serveur passe au chat template ; le texte : `.choices[0].message.content`
- Le prompt du serveur : `requests.post(f"{URL}/apply-template", json={"messages": conversation, "chat_template_kwargs": ...}).json()["prompt"]`, à comparer à `tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True, enable_thinking=False)`
- `serveur.terminate()`, puis `serveur.wait()` : arrête le serveur, et attend qu'il ait fini
- La ligne `transformers` : `evalue(sorties_export[:n])`, avec `n = len(serveur_part)` ; son temps par ticket est celui de l'exercice 2 (sur GPU, par lots de 16)

```python
def demarre_serveur(chemin: str) -> subprocess.Popen:
  journal = open("llama-server.log", "w")
  serveur = subprocess.Popen([...], stdout=journal, stderr=subprocess.STDOUT)
  while True:
    try:
      if requests.get(f"{URL}/health").status_code == 200:
        return serveur
    except requests.ConnectionError:
      pass
    if serveur.poll() is not None:
      raise RuntimeError("llama-server s'est arrêté : lisez llama-server.log")
    time.sleep(1)


def genere_serveur(conversations: list[list[dict]]) -> list[str]:
  textes = []
  for conversation in conversations:
    reponse = client.chat.completions.create(...)
    textes.append(reponse.choices[0].message.content)
  return textes


n = len(serveur_part)
conversations = [conversation_seule(m) for m in serveur_part["message"]]
lignes = {"transformers": evalue(sorties_export[:n])}
lignes["transformers"]["s_par_ticket"] = comparatif.loc["transformers", "s_par_ticket"]
lignes["transformers"]["taille_mo"] = comparatif.loc["transformers", "taille_mo"]
for nom in ["Q8_0", "Q4_K_M"]:
  serveur = demarre_serveur(ggufs[nom])
  prompt = requests.post(...).json()["prompt"]
  attendu = tokenizer.apply_chat_template(...)
  print(nom, "même prompt que transformers :", prompt == attendu)
  debut = time.time()
  sorties = genere_serveur(conversations)
  secondes = (time.time() - debut) / n
  serveur.terminate()
  serveur.wait()
  lignes[nom] = evalue(sorties)
  lignes[nom]["s_par_ticket"] = round(secondes, 2)
  lignes[nom]["taille_mo"] = round(taille_mo(ggufs[nom]))
  lignes[nom]["identiques"] = ...
lignes["transformers"]["identiques"] = 1.0
formats = pd.DataFrame(lignes).T
print(sorties[0])
formats
```

### Solution

In [ ]:
def demarre_serveur(chemin: str) -> subprocess.Popen:
  journal = open("llama-server.log", "w")
  serveur = subprocess.Popen(
    [
      f"{BINAIRES}/llama-server",
      "--model",
      chemin,
      "--port",
      str(PORT),
      "--alias",
      "support-sft",
      "--threads",
      str(CPUS),
    ],
    stdout=journal,
    stderr=subprocess.STDOUT,
  )
  while True:
    try:
      if requests.get(f"{URL}/health").status_code == 200:
        return serveur
    except requests.ConnectionError:
      pass
    if serveur.poll() is not None:
      raise RuntimeError("llama-server s'est arrêté : lisez llama-server.log")
    time.sleep(1)


def genere_serveur(conversations: list[list[dict]]) -> list[str]:
  textes = []
  for conversation in conversations:
    reponse = client.chat.completions.create(
      model="support-sft",
      messages=conversation,
      temperature=0,
      max_tokens=384,
      extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    textes.append(reponse.choices[0].message.content)
  return textes


n = len(serveur_part)
conversations = [conversation_seule(m) for m in serveur_part["message"]]
lignes = {"transformers": evalue(sorties_export[:n])}
lignes["transformers"]["s_par_ticket"] = comparatif.loc["transformers", "s_par_ticket"]
lignes["transformers"]["taille_mo"] = comparatif.loc["transformers", "taille_mo"]
for nom in ["Q8_0", "Q4_K_M"]:
  serveur = demarre_serveur(ggufs[nom])
  prompt = requests.post(
    f"{URL}/apply-template",
    json={
      "messages": conversations[0],
      "chat_template_kwargs": {"enable_thinking": False},
    },
  ).json()["prompt"]
  attendu = tokenizer.apply_chat_template(
    conversations[0], tokenize=False, add_generation_prompt=True, enable_thinking=False
  )
  print(nom, "même prompt que transformers :", prompt == attendu)
  debut = time.time()
  sorties = genere_serveur(conversations)
  secondes = (time.time() - debut) / n
  serveur.terminate()
  serveur.wait()
  lignes[nom] = evalue(sorties)
  lignes[nom]["s_par_ticket"] = round(secondes, 2)
  lignes[nom]["taille_mo"] = round(taille_mo(ggufs[nom]))
  lignes[nom]["identiques"] = sum(a == b for a, b in zip(sorties, sorties_export)) / n
lignes["transformers"]["identiques"] = 1.0
formats = pd.DataFrame(lignes).T
print(sorties[0])
formats

# Des sorties identiques sont rares, même en Q8_0 : il suffit d'un token qui change
# (un arrondi, un autre ordre de calcul, un prompt seul plutôt qu'en lot) pour que la
# génération gloutonne bifurque. C'est la mesure de la tâche qui dit ce que la
# quantification a coûté, sur assez de tickets : Q8_0 ne perd presque rien, Q4_K_M
# un peu plus, pour un fichier plus petit et plus rapide ; c'est le choix courant
# sur processeur, si la mesure le permet. Sur processeur, llama.cpp génère bien plus
# vite que transformers en float32. Le serveur garde en cache le début commun des
# prompts (la consigne) : seul le message du client est recalculé à chaque requête.

## 5. La fiche de traçabilité

Dans six mois, le modèle livré doit encore pouvoir dire d'où il vient : quelles données, quel modèle de base, quels réglages, quelles mesures, et si c'est bien ce fichier-là. Préparez le dossier de livraison, `LIVRAISON` (ci-dessous) : copiez-y le fichier GGUF que vous livrez, et écrivez-y sa carte de modèle, `README.md`, avec `ModelCardData` (l'en-tête YAML que lit le Hub : la langue, la licence `apache-2.0`, celle de Qwen3, le modèle de base, le jeu de données, des étiquettes) et `ModelCard`. Elle donne :

- les données : le jeu et sa version (`DONNEES`, ci-dessous) ;
- le modèle de base et sa révision, l'identifiant du commit téléchargé du Hub ;
- l'adaptateur : sa source, sa configuration LoRA (`adapter_config.json`) et les réglages de son entraînement (`ENTRAINEMENT`, ci-dessous : l'adaptateur de référence ne les porte pas) ;
- la livraison : la fusion, la conversion et la quantification, avec l'étiquette de llama.cpp ;
- les résultats : les mesures des exercices 2 et 4 ;
- l'empreinte SHA-256 des fichiers livrés, et de l'export safetensors, dont on repartira pour une autre quantification ;
- les versions des bibliothèques ;
- l'usage prévu et les limites.

Puis, si vous avez un compte Hugging Face, publiez le dossier sur le Hub, en **privé**, sous `<nom>/support-sft-gguf`. Il faut le jeton `HF_TOKEN` des TP précédents, dans les secrets de Colab ; sans lui, la cellule ne publie rien.

**Documentation utile**

- [`ModelCard` et `ModelCardData`](https://huggingface.co/docs/huggingface_hub/package_reference/cards)
- [`HfApi.model_info`](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.model_info)
- [`hashlib.file_digest`](https://docs.python.org/3/library/hashlib.html#hashlib.file_digest)
- [`shutil.copy`](https://docs.python.org/3/library/shutil.html#shutil.copy)
- [`HfApi.upload_folder`](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.upload_folder)

In [ ]:
LIVRAISON = "support-sft-gguf"
# Le jeu d'entraînement : le vôtre sur le Hub si l'adaptateur est le vôtre, sinon celui
# que le TP de données produit, publié dans shuuchuu/datasets.
DONNEES = {
  "jeu": f"{depot.split('/')[0]}/support-sft" if depot else "support-sft",
  "source": "le Hub" if depot else "shuuchuu/datasets, bike-shop-support/lab-results",
  "version": "v1.0",
  "exemples": "1 190 d'entraînement, 133 de validation",
  "test": "les 150 tickets de bike-shop-support/test.jsonl",
}
# Les réglages du TP d'entraînement, ceux de l'adaptateur de référence. Votre
# adaptateur porte les siens dans sa carte, ADAPTATEUR/README.md.
ENTRAINEMENT = dict(
  methode="QLoRA : le modèle de base en NF4, double quantification ; SFTTrainer de trl",
  learning_rate=2e-4,
  lr_scheduler_type="cosine",
  warmup_steps=0.1,
  per_device_train_batch_size=4,
  gradient_accumulation_steps=4,
  optim="paged_adamw_8bit",
  max_length=1024,
  epoques="1, puis 2 : le checkpoint de plus basse perte de validation des deux runs",
)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `os.makedirs(LIVRAISON, exist_ok=True)`, puis `shutil.copy(ggufs["Q4_K_M"], LIVRAISON)` (ou le Q8_0, selon l'exercice 4)
- `HfApi().model_info(MODELE).sha` : l'identifiant du commit courant du modèle sur le Hub, celui que `from_pretrained` a téléchargé
- L'empreinte d'un fichier : `hashlib.file_digest(f, "sha256").hexdigest()`, sur le fichier ouvert en binaire (`open(chemin, "rb")`)
- `json.load` sur `adapter_config.json` : `r`, `lora_alpha`, `lora_dropout`, `target_modules`
- `ModelCardData(language="fr", license="apache-2.0", base_model=MODELE, datasets=..., tags=[...], pipeline_tag="text-generation")`, `datasets` n'ayant que les jeux du Hub : `[DONNEES["jeu"]]` si l'adaptateur est le vôtre (`depot`), sinon `None` ; son `.to_yaml()` est l'en-tête, entre deux lignes `---`, en tête du texte de `ModelCard(texte)` ; `carte.save(chemin)` l'enregistre
- `pd.DataFrame.to_markdown()` : un tableau en Markdown
- Les versions : `torch.__version__`, `version("transformers")` (`importlib.metadata`, importé au début)…
- Le jeton et la publication, comme au TP d'entraînement : `api.create_repo(depot_livraison, private=True, exist_ok=True)`, puis `api.upload_folder(folder_path=LIVRAISON, repo_id=depot_livraison)`, avec `depot_livraison = f"{nom}/support-sft-gguf"` (`depot` est celui de l'adaptateur) ; sur GPU, l'envoi du fichier GGUF prend quelques minutes

```python
def empreinte(chemin: str) -> str:
  with open(chemin, "rb") as f:
    return hashlib.file_digest(f, ...).hexdigest()


os.makedirs(LIVRAISON, exist_ok=True)
quantification = "Q4_K_M"  # le choix de l'exercice 4
livre = shutil.copy(ggufs[quantification], LIVRAISON)
revision = HfApi().model_info(MODELE).sha
with open(os.path.join(ADAPTATEUR, "adapter_config.json")) as f:
  lora = json.load(f)
empreintes = {
  os.path.basename(livre): empreinte(livre),
  "model.safetensors (export)": empreinte(os.path.join(EXPORT, "model.safetensors")),
}
entete = ModelCardData(
  language="fr",
  license=...,
  base_model=MODELE,
  datasets=[DONNEES["jeu"]] if depot else None,
  tags=["gguf", "llama.cpp", "lora", "sft"],
  pipeline_tag="text-generation",
)
texte = f"""# support-sft-gguf

Qwen3 affiné pour le service client de Biclou & Cie (fictif) : ...

## Données
...
## Modèle de base
...
## Adaptateur
...
## Livraison
...
## Résultats
...
## Empreintes SHA-256
...
## Versions
...
## Usage prévu et limites
...
"""
carte = ModelCard(f"---\n{entete.to_yaml()}\n---\n\n{texte}")
carte.save(os.path.join(LIVRAISON, "README.md"))
print(carte.content)

# La publication : celle du TP d'entraînement, avec le dossier LIVRAISON.
```

### Solution

In [ ]:
def empreinte(chemin: str) -> str:
  with open(chemin, "rb") as f:
    return hashlib.file_digest(f, "sha256").hexdigest()


os.makedirs(LIVRAISON, exist_ok=True)
quantification = "Q4_K_M"  # le choix de l'exercice 4
livre = shutil.copy(ggufs[quantification], LIVRAISON)
revision = HfApi().model_info(MODELE).sha
with open(os.path.join(ADAPTATEUR, "adapter_config.json")) as f:
  lora = json.load(f)
empreintes = {
  os.path.basename(livre): empreinte(livre),
  "model.safetensors (export)": empreinte(os.path.join(EXPORT, "model.safetensors")),
  "adapter_model.safetensors": empreinte(
    os.path.join(ADAPTATEUR, "adapter_model.safetensors")
  ),
}
entete = ModelCardData(
  language="fr",
  license="apache-2.0",
  base_model=MODELE,
  datasets=[DONNEES["jeu"]] if depot else None,
  tags=["gguf", "llama.cpp", "lora", "sft"],
  pipeline_tag="text-generation",
)
source = REFERENCE
if depot:
  source = f"[{depot}](https://huggingface.co/{depot}) : sa carte dit l'entraînement"
reglages = "\n".join(f"- `{cle}` : {valeur}" for cle, valeur in ENTRAINEMENT.items())
if depot:
  reglages = "- L'entraînement : celui que donne la carte de l'adaptateur"
liste_empreintes = "\n".join(f"- `{nom}` : `{h}`" for nom, h in empreintes.items())
texte = f"""# support-sft-gguf

{MODELE} affiné pour le service client de Biclou & Cie (fictif), fusionné et
quantifié en GGUF {quantification} pour llama.cpp : au message
d'un client, avec la consigne courte du TP (en message system), il répond par un objet
JSON `{{"categorie", "priorite", "reponse"}}` selon la charte du service client.

## Données

`{DONNEES["jeu"]}` ({DONNEES["source"]}), version `{DONNEES["version"]}` :
{DONNEES["exemples"]} ; mesuré sur {DONNEES["test"]}.

## Modèle de base

[{MODELE}](https://huggingface.co/{MODELE}), révision `{revision}`, licence Apache 2.0.

## Adaptateur

{source}

- LoRA : rang {lora["r"]}, `lora_alpha` {lora["lora_alpha"]}, dropout
  {lora["lora_dropout"]}, sur {", ".join(sorted(lora["target_modules"]))}
{reglages}

## Livraison

- Adaptateur fusionné (`merge_and_unload` de peft) dans le modèle de base en
  {str(DTYPE).removeprefix("torch.")}, enregistré en safetensors
- Converti en GGUF F16 par `convert_hf_to_gguf.py`, puis quantifié en
  {quantification} par `llama-quantize`, llama.cpp `{LLAMA_CPP}`
- Prompt : le chat template de Qwen3, embarqué dans le fichier, la réflexion
  désactivée (`chat_template_kwargs={{"enable_thinking": false}}`), température 0

## Résultats

Export safetensors, avec transformers, sur les {len(test_part)} premiers tickets :

{comparatif.to_markdown()}

Sur les {len(serveur_part)} premiers, servis par `llama-server` sur processeur :

{formats.to_markdown()}

## Empreintes SHA-256

{liste_empreintes}

## Versions

torch {torch.__version__}, transformers {version("transformers")}, peft
{version("peft")}, gguf {version("gguf")}, llama.cpp {LLAMA_CPP}

## Usage prévu et limites

Trier et rédiger une première réponse aux messages des clients, relue par un
conseiller avant envoi. Le modèle n'a appris que la forme des réponses de l'équipe :
il ne connaît pas les commandes, les stocks ni les délais du moment, et peut donner
un délai ou une condition faux. Hors du service client, il répond encore par un
ticket en JSON. À réévaluer à chaque changement de la charte ou des données.
"""
carte = ModelCard(f"---\n{entete.to_yaml()}\n---\n\n{texte}")
carte.save(os.path.join(LIVRAISON, "README.md"))
print(carte.content)

jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
if jeton:
  api = HfApi(token=jeton)
  depot_livraison = f"{api.whoami()['name']}/support-sft-gguf"
  api.create_repo(depot_livraison, private=True, exist_ok=True)
  api.upload_folder(folder_path=LIVRAISON, repo_id=depot_livraison)
  print("Publié :", depot_livraison)
else:
  print("Pas de jeton HF_TOKEN : le modèle et sa carte restent dans", LIVRAISON)

## À retenir

- Fusionner l'adaptateur donne un modèle ordinaire, aussi rapide que le modèle de base, que tout outil charge sans `peft` ; on fusionne dans des poids en 16 bits, jamais dans le modèle quantifié de QLoRA. On garde l'adaptateur séparé pour servir plusieurs adaptateurs sur un même modèle de base.
- L'export safetensors doit se suffire à lui-même : on le recharge seul et on le mesure. Le chat template voyage avec lui : sans lui, le modèle reçoit un prompt qu'il n'a jamais vu.
- GGUF met en un fichier les poids, le tokenizer et le chat template ; la quantification après l'entraînement (Q8_0, Q4_K_M) divise la taille par deux ou par trois et accélère le processeur, au prix de sorties qui s'écartent parfois : la mesure de la tâche dit si ce prix est acceptable.
- `llama-server` sert le modèle par l'API d'OpenAI : une application n'a qu'à changer d'adresse.
- La fiche de traçabilité relie le fichier livré, par son empreinte, aux données et à leur version, au modèle de base et à sa révision, aux réglages, aux mesures et aux versions des outils : c'est elle qui permet de refaire, de comparer et de réentraîner.